In [3]:
# Step 1: Install Required Libraries
# * `faiss-cpu`: For high-speed vector similarity matching.
# * `sentence-transformers`: For converting text into mathematical embeddings.
# * `g4f`: For communicating with the AI model.

In [4]:
!pip install faiss-cpu sentence-transformers g4f

In [ ]:
# Step 2: Import Modules into Memory

In [5]:
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
import g4f
from google.colab import files as colab_files
import os

In [6]:
# Step 3: Upload Your Knowledge Base Files
# You need to provide the text documents (`.txt`) that the AI will use to look up answers.

In [7]:
print("Please upload your knowledge base text files")
uploaded = colab_files.upload()

# Save the uploaded files to the local directory
uploaded_file_paths = []
for file_name in uploaded.keys():
    print(f"Successfully uploaded: {file_name}")
    uploaded_file_paths.append(file_name)

Please upload your knowledge base text files


Saving knowledgeA.txt to knowledgeA.txt
Successfully uploaded: knowledgeA.txt


In [8]:
# Step 4: Load the Embedding Model
# This cell initializes the AI model (`all-MiniLM-L6-v2`) responsible for reading your text sentences and converting them into numerical representations (embeddings).

In [9]:
# Initialize the embedding encoder model
encoder = SentenceTransformer("all-MiniLM-L6-v2")
print("Embedding model loaded successfully.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully.


In [10]:
# Step 5: Index the Document Text
# This step extracts all individual lines of text from the files you uploaded in Step 3, removes empty spaces, and injects them into a high-performance **FAISS Vector Index** matrix. This allows the system to instantly search through thousands of lines structurally.

In [11]:
documents = []

# Verify files exist and load text lines
if 'uploaded_file_paths' in locals() and uploaded_file_paths:
    for path in uploaded_file_paths:
        if os.path.exists(path):
            with open(path, "r", encoding="utf-8") as file:
                documents.extend([line.strip() for line in file if line.strip()])
        else:
            print(f"Warning: File {path} was not found.")

    if not documents:
        print("Error: All uploaded files are empty!")
    else:
        print(f"Successfully loaded {len(documents)} raw document lines from your files.")
else:
    print("Error: Please run the upload cell (Cell 3) first!")

Successfully loaded 4 raw document lines from your files.


In [12]:
# Vectorize documents
embeddings = encoder.encode(documents, convert_to_numpy=True).astype('float32')
dimension = embeddings.shape[1]

# Build and populate the FAISS search index
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

print(f"Vector Index established. Total indexed documents: {index.ntotal}")

Vector Index established. Total indexed documents: 4


In [13]:
# Step 6: Define RAG Operational Logic
# This cell registers the core pipeline functions behind the scenes

In [14]:
def retrieve(query: str, top_k: int = 1) -> str:
    # Vectorize incoming query
    query_vector = encoder.encode([query], convert_to_numpy=True).astype('float32')
    distances, indices = index.search(query_vector, top_k)
    best_match_idx = indices[0][0]

    if best_match_idx != -1:
        return documents[best_match_idx]
    return "[No secure context found]"

def augment_and_generate(query: str, context: str) -> str:
    prompt = f"""You are a precise database helper. Answer the user's question using ONLY the provided Context block.

Rules:
1. Provide a direct, plain answer based on the context.
2. Do NOT say "Based on the context...", do NOT explain your reasoning, and do NOT talk about the prompt.
3. If the question asks "what is [X]" and the context says "[Y] is [X]", answer with "[Y]".
4. If the context does not contain the answer, reply exactly with: "I don't know."

Context:
{context}

User Question: {query}
Answer:"""

    response = g4f.ChatCompletion.create(
        model=g4f.models.default,
        messages=[{"role": "user", "content": prompt}],
    )
    return response

In [15]:
# Step 7: Launch the Chat Interface

In [16]:
def query_pipeline(user_query: str) -> str:
    try:
        # Step 1: Find best vector match
        context = retrieve(user_query, top_k=1)
        # Step 2: Route augmented prompt to LLM
        output = augment_and_generate(user_query, context)
        return output
    except Exception as error:
        return f"Error encountered: {str(error)}"

In [17]:
print("RAG System Active! Type 'exit' to quit out of the prompt loops.")

while True:
    user_input = input("\nEnter your query: ")
    if user_input.strip().lower() == 'exit':
        print("System execution halted.")
        break

    if not user_input.strip():
        continue

    final_response = query_pipeline(user_input)
    print(f"[System Output]: {final_response}")

RAG System Active! Type 'exit' to quit out of the prompt loops.

Enter your query: tell me about apple
g4f is up-to-date (version 7.5.5).
[System Output]: Apple

Enter your query: exit
System execution halted.
